# AI-Powered Data Hygiene Agent

## Overview
This agent automatically analyzes B2B lead records and identifies data quality issues 
that would impact marketing operations, lead routing, and CRM integrity.

## Issues Detected
- **Duplicate Records** — same email across multiple leads
- **Invalid Emails** — missing domain extensions, empty fields
- **Test Records** — mailinator, +test aliases that should never reach production
- **Missing Fields** — empty company, title, or email fields
- **Inconsistent Data** — country format variations (US/USA/United States), 
  name casing, company name variations

## Output Format
For each issue the agent returns:
- **Lead ID** — which record is affected
- **Issue Type** — category of the problem
- **Details** — specific explanation
- **Recommendation** — merge / quarantine / enrich / normalize / delete

## Real-World Application
In a production MarTech stack this agent would:
1. Run on a schedule against new leads entering Marketo or Salesforce
2. Flag issues before they impact lead scoring or routing
3. Trigger enrichment workflows for incomplete records
4. Auto-merge duplicates based on source trust priority rules

## Tech Stack
- Python
- Anthropic Claude API (claude-haiku)
- Jupyter Notebook

## Author
Karthik Ayyer — Marketing Automation Manager| Marketing Operations Manager | MarTech & Revenue Operations

In [3]:
!pip install anthropic python-dotenv -q

In [7]:
from dotenv import load_dotenv
import os
import anthropic

load_dotenv()
client = anthropic.Anthropic()

In [8]:
leads = [
    {"id": 1, "name": "Rahul Sharma", "email": "rahul@techcorp.com", "company": "TechCorp India", "country": "India", "title": "VP of Marketing"},
    {"id": 2, "name": "rahul sharma", "email": "rahul@techcorp.com", "company": "Techcorp india", "country": "IN", "title": "VP Marketing"},
    {"id": 3, "name": "Priya Patel", "email": "priya@startup.com", "company": "StartupXYZ", "country": "US", "title": "Manager"},
    {"id": 4, "name": "James Wilson", "email": "james@enterprise", "company": "Enterprise Corp", "country": "USA", "title": "CMO"},
    {"id": 5, "name": "Anita Rao", "email": "", "company": "MidMarket Inc", "country": "United States", "title": "Director"},
    {"id": 6, "name": "David Lee", "email": "david@smallbiz.com", "company": "", "country": "GB", "title": ""},
    {"id": 7, "name": "Priya Patel", "email": "priya@startup.com", "company": "StartupXYZ", "country": "US", "title": "Marketing Manager"},
    {"id": 8, "name": "test user", "email": "test+alias@mailinator.com", "company": "Test Co", "country": "US", "title": "Tester"},
]

In [9]:
hygiene_prompt = """
You are a MarTech data quality expert. Analyze this list of leads and identify all data quality issues.

For each issue found, flag it with:
- LEAD ID: which lead has the issue
- ISSUE TYPE: Duplicate / Invalid Email / Missing Field / Inconsistent Data / Test Record
- DETAILS: specific explanation of the problem
- RECOMMENDATION: what action to take (merge, quarantine, enrich, normalize)

Be thorough and check for:
1. Duplicate emails across records
2. Invalid email formats
3. Missing required fields (email, company, title)
4. Inconsistent country formats
5. Test or fake emails
6. Name and company casing inconsistencies
"""

In [14]:
message = client.messages.create(
    model ="claude-haiku-4-5-20251001",
    max_tokens=2048,
    messages=[
        {
            "role":"user",
            "content": f"{hygiene_prompt}\n\nHere are the leads to analyze:\n{leads}"
        }
    ]
)
print(message.content[0].text)

# MarTech Data Quality Analysis Report

## Summary
**Total Leads Analyzed:** 8
**Issues Found:** 11
**Critical Issues:** 5

---

## Detailed Issues by Lead

### 🔴 LEAD ID: 1 & 2
**ISSUE TYPE:** Duplicate + Inconsistent Data
**DETAILS:** 
- Same email address (rahul@techcorp.com) across two records
- Lead 1: "Rahul Sharma" | "TechCorp India" | "India" | "VP of Marketing"
- Lead 2: "rahul sharma" (lowercase) | "Techcorp india" (mixed case) | "IN" (country code vs full name) | "VP Marketing" (abbreviated)
- Identical core data with formatting inconsistencies

**RECOMMENDATION:** MERGE - Keep Lead 1 as master record (proper formatting). Delete Lead 2.

---

### 🔴 LEAD ID: 4
**ISSUE TYPE:** Invalid Email
**DETAILS:** Email format incomplete - "james@enterprise" lacks domain extension (missing TLD like .com, .org, etc.)

**RECOMMENDATION:** QUARANTINE - Flag for manual verification or outreach to confirm correct email domain.

---

### 🔴 LEAD ID: 5
**ISSUE TYPE:** Missing Field
**DETAILS:** 